# KOVA Speech-to-Text — Google Colab GPU Worker

This is a dedicated Faster-Whisper worker for **KOVA source speech-to-text**. It is separate from KOVA Voice Studio / OmniVoice.

1. Select **Runtime → Change runtime type → GPU**.
2. Select **Runtime → Run all**.
3. Copy `KOVA_STT_URL` and `KOVA_STT_TOKEN` from the last cell into KOVA’s **Nguồn video → Speech-to-text** panel.

The URL and token expire when this Colab runtime stops. The worker rejects CPU fallback and accepts only authenticated requests.

In [ ]:
import os, subprocess, sys

def run(command):
    print('+', ' '.join(map(str, command)))
    subprocess.run(command, check=True)

run(['nvidia-smi'])
run([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade',
     'faster-whisper==1.1.1', 'fastapi==0.115.12', 'uvicorn[standard]==0.34.2',
     'python-multipart==0.0.20', 'requests==2.32.3'])
print('CUDA runtime detected. Installing/loading the Medium model happens in the next cell.')

In [ ]:
from pathlib import Path

WORKER = Path('/content/kova_stt_worker.py')
WORKER.write_text(r'''
import os
import secrets
import shutil
from pathlib import Path
from typing import Optional

import ctranslate2
from fastapi import Depends, FastAPI, File, Form, Header, HTTPException, UploadFile
from faster_whisper import WhisperModel

TOKEN = os.environ['KOVA_STT_API_TOKEN']
MODEL_NAME = os.environ.get('KOVA_STT_MODEL', 'medium')
MODEL_DIR = os.environ.get('KOVA_STT_MODEL_DIR', '/content/kova-stt-models')
UPLOAD_DIR = Path('/content/kova-stt-uploads')
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

if ctranslate2.get_cuda_device_count() < 1:
    raise RuntimeError('CUDA is unavailable. Stop and select a GPU runtime; CPU fallback is disabled.')

# Loading here makes /health truthful: KOVA can only start after both CUDA
# and the selected public Faster-Whisper model are ready.
MODEL = WhisperModel(MODEL_NAME, device='cuda', compute_type='float16', download_root=MODEL_DIR)
app = FastAPI(title='KOVA Colab STT Worker', docs_url=None, redoc_url=None, openapi_url=None)

def authorize(authorization: str = Header(default='')):
    expected = 'Bearer ' + TOKEN
    if not secrets.compare_digest(authorization, expected):
        raise HTTPException(status_code=401, detail='invalid or missing bearer token')

@app.get('/health')
def health(_: None = Depends(authorize)):
    return {'ready': True, 'device': 'cuda', 'model': MODEL_NAME}

@app.post('/v1/audio/transcriptions')
async def transcribe(
    file: UploadFile = File(...),
    model: str = Form(default='medium'),
    response_format: str = Form(default='verbose_json'),
    language: Optional[str] = Form(default=None),
    timestamp_granularities: list[str] = Form(default=[]),
    _: None = Depends(authorize),
):
    # KOVA needs verbose JSON with word timestamps. The worker uses the
    # model selected by this notebook, not an untrusted client model id.
    if response_format not in ('verbose_json', 'json'):
        raise HTTPException(status_code=400, detail='response_format must be verbose_json or json')
    suffix = Path(file.filename or 'audio.mp3').suffix or '.mp3'
    upload = UPLOAD_DIR / (secrets.token_urlsafe(16) + suffix)
    try:
        with upload.open('wb') as output:
            shutil.copyfileobj(file.file, output)
        segments_iter, info = MODEL.transcribe(
            str(upload), language=(language or None), beam_size=5,
            word_timestamps=True, vad_filter=True, condition_on_previous_text=True,
        )
        segments = []
        words = []
        for index, segment in enumerate(segments_iter):
            text = segment.text.strip()
            segments.append({
                'id': index, 'seek': 0, 'start': segment.start, 'end': segment.end,
                'text': text, 'tokens': [], 'temperature': 0, 'avg_logprob': 0,
                'compression_ratio': 0, 'no_speech_prob': 0,
            })
            for word in segment.words or []:
                words.append({'word': word.word, 'start': word.start, 'end': word.end})
        return {
            'text': ' '.join(item['text'] for item in segments if item['text']),
            'language': info.language, 'duration': info.duration,
            'segments': segments, 'words': words,
        }
    finally:
        await file.close()
        upload.unlink(missing_ok=True)
''', encoding='utf-8')
print('Worker source prepared:', WORKER)

In [ ]:
import json, secrets, time, urllib.request

TOKEN = secrets.token_urlsafe(32)
ENV = os.environ.copy()
ENV.update({
    'KOVA_STT_API_TOKEN': TOKEN,
    'KOVA_STT_MODEL': 'medium',
    'KOVA_STT_MODEL_DIR': '/content/kova-stt-models',
})
LOG = '/content/kova-stt-worker.log'
worker = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'kova_stt_worker:app', '--host', '127.0.0.1', '--port', '3940'], cwd='/content', env=ENV, stdout=open(LOG, 'w'), stderr=subprocess.STDOUT)

for _ in range(150):
    try:
        request = urllib.request.Request('http://127.0.0.1:3940/health', headers={'Authorization': 'Bearer ' + TOKEN})
        with urllib.request.urlopen(request, timeout=5) as response:
            health = json.load(response)
        if health.get('ready') and health.get('device') == 'cuda':
            break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError('STT worker did not become CUDA-ready. Log tail:\n' + Path(LOG).read_text(errors='replace')[-4000:])

run(['wget', '-q', '-O', '/content/cloudflared.deb', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb'])
run(['dpkg', '-i', '/content/cloudflared.deb'])
tunnel = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:3940', '--no-autoupdate'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
public_url = None
for _ in range(90):
    line = tunnel.stdout.readline()
    print(line, end='')
    if 'https://' in line and 'trycloudflare.com' in line:
        public_url = line[line.find('https://'):].split()[0]
        break
if not public_url:
    raise RuntimeError('Cloudflare tunnel URL was not found.')

print('\nKOVA_STT_URL=' + public_url)
print('KOVA_STT_TOKEN=' + TOKEN)
print('MODEL=medium; DEVICE=cuda')
print('Copy the URL and token into KOVA > Nguồn video > Speech-to-text. Do not add /v1 to the URL.')